In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('QtAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd

from scipy.stats import permutation_test

from statsmodels.tsa.stattools import acf

sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5
from more_itertools import collapse
from joblib import Parallel, delayed


In [2]:
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
if modality=="auditory":
    subj="sub-A2002"
else:
    subj = "sub-V1001"

# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

In [3]:

#epochs
combinaciones = ["zinnen", "woorden"]

subjects=[]

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)

print(subjects)


##tablas de canales

channels_path = channels_structure_path   / f"channels_mag_{modality}.csv"

channels = pd.read_csv(channels_path)
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]
channels_mag=channels_mag.tolist()
del channels
print(len(channels_mag))

# epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subjects[0]}_epochs_zinnen_{layer_script}-epo.fif")
window_size=3
sliding_window=0.351


##values for function
# ## MARK THEM FOR FUNCTION CALL
# condition="zinnen"
# adjusted=False
# fft=True
# alpha=None
# bartlett_confint=True
# missing="none"
# isplot=False

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [4]:

def dynamic_acf_epochs(subj, epochs,condition,window_size,sliding_window,channels_mag,adjusted=False,fft=True,alpha=None, bartlett_confint=True, missing="none",isplot=False): 

    if type(epochs)== mne.epochs.EpochsFIF:
        data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()

    #import epochs object and get data ONLY ON MEG DATA and ALSO EXCLUDE BADS

    if type(epochs) == np.ndarray:
        data_epochs = epochs

    #get duration
    duration= epochs.tmax - epochs.tmin
    #get sample frequency
    sfreq= epochs.info['sfreq']
    #get lags
    lags=duration*sfreq

    # Convertir a muestras
    sliding_window_samples = int(sliding_window * sfreq)
    window_size_samples = int(window_size * sfreq)

    #list to store the values of acf, acw_50,acw_0 for each epoch for each sensor for each EPOCH
    # acf_window_all_elect_all_epoch_all = []

    #conjunto de todos los valores de ACW_0-ACW_50 para cada sensor en cada epoca
    acw_50_window_all_elect_all_epoch_all = []
    acw_0_window_all_elect_all_epoch_all = []
    
    acw_50_slope_elect_all_epoch_all=[]
    acw_50_std_elect_all_epoch_all=[]
    
    acw_0_slope_elect_all_epoch_all=[]
    acw_0_std_elect_all_epoch_all=[]
    


    ##for each epoch  
    for j in range(0,len(data_epochs)):

        #take data for each epoch
        data_epoch=data_epochs[j]

        #list to store the values of acf, acw_50,acw_0 for each SENSOR for each EPOCH

        # acf_window_all_elect_all_epoch = []
        acw_50_window_all_elect_all_epoch = []
        acw_0_window_all_elect_all_epoch = []
        
        acw_50_slope_elect_all_epoch=[]
        acw_50_std_elect_all_epoch=[]
        
        acw_0_slope_elect_all_epoch=[]
        acw_0_std_elect_all_epoch=[]
        
        #calculus of value  for EACH SENSOR
        for i in range(0,len(data_epoch)):
        ##for i in range(0,1):
            #lags se puede dejar por defecto porque te cogelos valores
            #hasta los que tiene sentido calcularlo, aunque yo voy a calcularlo para todos los lags
            
            
            #calculo de acf de sensor
            #alfa lo dejo endefault que es la confianza del 95% 
            
            ###break epochs in segments of size window_size_samples with a step of sliding_window_samples
            epoch = data_epoch[i]
            n_samples = epoch.shape[0]
            
            #el acw_50 y acw_0 se calculan en cada ventana de la epoca, and should be SAME lenght that 

            # acf_window_all_elect_epoch = []
            acw_50_window_all_elect_epoch = []
            acw_0_window_all_elect_epoch = []
            
                # 1️⃣ Definir función para procesar cada ventana individual
            def compute_acf_fit(segment, adjusted, fft, lags, alpha, bartlett_confint, missing, sfreq):
                acf_window_elect_epoch, _, _ = acf(
                segment,
                adjusted=adjusted,
                fft=fft,
                qstat=True,
                nlags=lags,
                alpha=alpha,
                bartlett_confint=bartlett_confint,
                missing=missing
                )

                acw_50_lags = np.argmax(acf_window_elect_epoch <= 0.5)
                acw_0_lags = np.argmax(acf_window_elect_epoch <= 0)

                acw_50_s = acw_50_lags / sfreq
                acw_0_s = acw_0_lags / sfreq

                return acw_50_s, acw_0_s
            
                            # 2️⃣ Crear lista de segmentos válidos
            segments = [
                epoch[start:start + window_size_samples]
                for start in range(0, n_samples - window_size_samples + 1, sliding_window_samples)
                if epoch[start:start + window_size_samples].shape[0] == window_size_samples
            ]

            window_number = len(segments)
            
                        # 3️⃣ Procesar en paralelo con joblib
            window_results = Parallel(n_jobs=-1)(
                delayed(compute_acf_fit)(
                    segment, adjusted, fft, lags, alpha, bartlett_confint, missing, sfreq
                )
                for segment in segments
            )

            # 4️⃣ Desempaquetar resultados
            for acw_50_s, acw_0_s in window_results:
                acw_50_window_all_elect_epoch.append(acw_50_s)
                acw_0_window_all_elect_epoch.append(acw_0_s)

                    
            ## ADD RESULT TO ALL SENSORS
            ##acf_window_all_elect_all_epoch.append(acf_window_all_elect_epoch)
            acw_50_window_all_elect_all_epoch.append(acw_50_window_all_elect_epoch)
            acw_0_window_all_elect_all_epoch.append(acw_0_window_all_elect_epoch)
            
            x=range(0,len(acw_50_window_all_elect_epoch))
            y=acw_50_window_all_elect_epoch
            coeffs_acw_50=np.polyfit(x,y, deg=1)
            acw_50_slope = coeffs_acw_50[0]
            acw_50_std = np.std(acw_50_window_all_elect_epoch)
            acw_50_slope_elect_all_epoch.append(acw_50_slope)
            acw_50_std_elect_all_epoch.append(acw_50_std)

            x=range(0,len(acw_0_window_all_elect_epoch))
            y=acw_0_window_all_elect_epoch
            coeffs_acw_0=np.polyfit(x,y, deg=1)
            acw_slope_0= coeffs_acw_0[0]
            acw_0_std = np.std(acw_0_window_all_elect_epoch)
            acw_0_slope_elect_all_epoch.append(acw_slope_0)
            acw_0_std_elect_all_epoch.append(acw_0_std)

        
        ## ADD RESULT TO ALL EPOCHS
    #     #conjunto de todos los valores de ACW_0-ACW_50 para cada sensor en cada epoca
    #     #acf_window_all_elect_all_epoch_all.append(acf_window_all_elect_all_epoch)
        acw_50_window_all_elect_all_epoch_all.append(acw_50_window_all_elect_all_epoch)
        acw_0_window_all_elect_all_epoch_all.append(acw_0_window_all_elect_all_epoch)
        
        acw_50_slope_elect_all_epoch_all.append(acw_50_slope_elect_all_epoch)
        acw_50_std_elect_all_epoch_all.append(acw_50_std_elect_all_epoch)
        acw_0_slope_elect_all_epoch_all.append(acw_0_slope_elect_all_epoch)
        acw_0_std_elect_all_epoch_all.append(acw_0_std_elect_all_epoch)
        

    # parameters of table
    num_elects=len(channels_mag)
    num_epochs=len(data_epochs)
    shape_tabla_dynamic=num_epochs*num_elects*window_number
    shape_tabla_results=num_epochs*num_elects

    #     COMO ACF CONTIENE UNA SERIE TEMPORAL , sus dimensiones son (n_epochs, nchans, acf_series), Y quiero meterla con el resto de condiciones (nchans, n_epochs)
    # para que las dimensiones cuadren, la rehsape en n_epochs*nchans, acf_series, y lo transformo en una lista, para que cada fila sea una lista y así poder meterlo en el dataframe 

    # # acf_elect_all_epoch_all_list = np.array(acf_elect_all_epoch_all).reshape(num_epochs * num_elects, np.array(acf_elect_all_epoch_all).shape[2]).tolist()
    table_dynamic_autocorrelation = pd.DataFrame({
        'Subject': [subj] * shape_tabla_dynamic,
        'Condition': [condition] * shape_tabla_dynamic,
        'Epoch': np.repeat(np.arange(num_epochs), num_elects * window_number),
        'Elect': np.tile(np.repeat(channels_mag, window_number), num_epochs),
        'Window': np.tile(np.arange(window_number), num_epochs * num_elects),
        'acw_50_window_all_elect_all_epoch_all': np.array(acw_50_window_all_elect_all_epoch_all).flatten(),    
        'acw_0_window_all_elect_all_epoch_all': np.array(acw_0_window_all_elect_all_epoch_all).flatten()

    })
    
    table_dynamic_autocorrelation_results = pd.DataFrame({
        'Subject': [subj] * shape_tabla_results,
        'Condition': [condition] * shape_tabla_results,
        'Epoch': np.repeat(np.arange(num_epochs), num_elects),
        'Elect': np.tile(channels_mag, num_epochs),
        'acw_50_slope_elect_all_epoch_all': np.array(acw_50_slope_elect_all_epoch_all).flatten(),    
        'acw_50_std_elect_all_epoch_all': np.array(acw_50_std_elect_all_epoch_all).flatten(),
        'acw_0_slope_elect_all_epoch_all': np.array(acw_0_slope_elect_all_epoch_all).flatten(),    
        'acw_0_std_elect_all_epoch_all': np.array(acw_0_std_elect_all_epoch_all).flatten()
        
    })
    
    return table_dynamic_autocorrelation, table_dynamic_autocorrelation_results

    
    


In [5]:
#table_dynamic_autocorrelation

In [6]:
# canal_buscado = 'MLF31-4304'

# if canal_buscado in channels_mag:
#     indice = channels_mag.index(canal_buscado)
#     print(f"El canal {canal_buscado} está en la posición {indice}.")
# else:
#     print(f"El canal {canal_buscado} NO está en la lista de channels_mag.")

In [7]:
##codigo para agrupar todas las tablas
####def dynamic_acf_epochs(subj, epochs,condition,window_size,sliding_window,channels_mag,adjusted=False,fft=True,alpha=None, bartlett_confint=True, missing="none",isplot=False): 

all_tables_dynamic = []
all_tables_dynamic_results=[]
for i in range(0,len(subjects)):
#for i in range(0,1):   
    for h in range(0,len(combinaciones)):
        try:
            subj=subjects[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subj}_epochs_{combinacion}_{layer_script}-epo.fif"
            epochs = mne.read_epochs(path_epochs)
            table_dynamic_autocorrelation, table_dynamic_autocorrelation_results= dynamic_acf_epochs(subj, epochs,combinacion,window_size, sliding_window,channels_mag,isplot=False)
            all_tables_dynamic.append(table_dynamic_autocorrelation)
            all_tables_dynamic_results.append(table_dynamic_autocorrelation_results)
            del epochs
        except Exception as e:
            print(f"Error en {subj} {combinacion}: {e}")

dynamic_autocorrelation_subjects_all = pd.concat(all_tables_dynamic, ignore_index=True)
dynamic_autocorrelation_results_subjects_all = pd.concat(all_tables_dynamic_results, ignore_index=True)



Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1001_epochs_zinnen_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
118 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_4656\4115333315.py:4: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1001_epochs_woorden_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
94 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_4656\4115333315.py:4: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1002_epochs_zinnen_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
112 matching events found
No baseline correction applied
0 projection items activated


...

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1031_epochs_zinnen_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
118 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_4656\4115333315.py:4: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1031_epochs_woorden_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
100 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_4656\4115333315.py:4: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


Error en sub-V1032 zinnen: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1032_epochs_zinnen_event-epo.fif"
Error en sub-V1032 woorden: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1032_epochs_woorden_event-epo.fif"
Error en sub-V1033 zinnen: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1033_epochs_zinnen_event-epo.fif"
Error en sub-V1033 woorden: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1033_epochs_woorden_event-epo.fif"
Error en sub-V1034 zinnen: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1034_epochs_zinnen_event-epo.fif"
Error en sub-V1034 woorden: File does not exist: "g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1034_epochs_woorden_event-epo.fif"
Error en sub-V1035 zinnen: File do

In [8]:
dynamic_autocorrelation_subjects_all.to_pickle(ACW_path / f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle")
dynamic_autocorrelation_results_subjects_all.to_pickle(ACW_path / f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle")